# Just predict: Miami housing and website phishing
Run top to bottom with the Exercise 2 dependencies installed (`pandas`, `pyarrow`, `requests`, `python-dotenv`). All prediction code is here. Credentials should be stored within `../../credentials/.env` file.

Each request uses 128 labelled examples and predicts five other rows. These small samples illustrate the API. `x_test` never contains the target; predictions follow its row order.


In [ ]:
import json
import os

import pandas as pd
import requests
from dotenv import load_dotenv

load_dotenv("../credentials/.env");


In [2]:
def request_body(dataset_id, target, task):
    data = pd.read_parquet(
        f"https://data.openml.org/datasets/0004/{dataset_id}/dataset_{dataset_id}.pq"
    ).dropna(subset=[target])
    # Keep identical feature patterns out of both context and query.
    data = data.drop_duplicates(subset=[c for c in data if c != target])
    sample = data.sample(n=133, random_state=42)
    context, query = sample.iloc[:128], sample.iloc[128:]
    features = [c for c in data if c != target]
    return {
        "task_config": {
            "task": task,
            "predict_params": {"output_type": "mean" if task == "regression" else "preds"},
            "tabpfn_config": {"random_state": 42},
        },
        "x_train": {c: json.loads(context[c].to_json(orient="values")) for c in features},
        "y_train": {target: json.loads(context[target].to_json(orient="values"))},
        "x_test": {c: json.loads(query[c].to_json(orient="values")) for c in features},
    }


### Miami house prices (USD)
[OpenML 46942](https://www.openml.org/search?type=data&id=46942), CC BY-NC-SA. Bourassa et al., *Big data, accessibility and urban house prices* (2021).


In [3]:
miami_body = request_body(46942, "SALE_PRC", "regression")
print(json.dumps(miami_body, indent=2))


{
  "task_config": {
    "task": "regression",
    "predict_params": {
      "output_type": "mean"
    },
    "tabpfn_config": {
      "random_state": 42
    }
  },
  "x_train": {
    "LATITUDE": [
      25.5351730241,
      25.713524078,
      25.7635571826,
      25.5720423651,
      25.8892652192,
      25.5159092007,
      25.6903569185,
      25.571883167,
      25.7047782821,
      25.9473028762,
      25.9562959231,
      25.5577143999,
      25.9483570183,
      25.744302997,
      25.6200578125,
      25.9617191119,
      25.9494148775,
      25.8867402353,
      25.9066393438,
      25.6094250146,
      25.5669853366,
      25.8717283162,
      25.8555525554,
      25.5600110471,
      25.5778960537,
      25.7469076123,
      25.7429167772,
      25.7383079973,
      25.9559786867,
      25.9480164964,
      25.9123749452,
      25.5854624843,
      25.5512683523,
      25.6384685877,
      25.5823164246,
      25.844153663,
      25.7310453487,
      25.4663784297,
      25

### Website phishing (class label)
[OpenML 46963](https://www.openml.org/search?type=data&id=46963), CC BY 4.0. Abdelhamid et al., *Phishing detection based associative classification data mining* (2014).


In [4]:
phishing_body = request_body(46963, "WebsiteType", "classification")
print(json.dumps(phishing_body, indent=2))


{
  "task_config": {
    "task": "classification",
    "predict_params": {
      "output_type": "preds"
    },
    "tabpfn_config": {
      "random_state": 42
    }
  },
  "x_train": {
    "SFH": [
      "Legitimate",
      "Legitimate",
      "Suspicious",
      "Phishy",
      "Suspicious",
      "Phishy",
      "Phishy",
      "Legitimate",
      "Legitimate",
      "Legitimate",
      "Suspicious",
      "Phishy",
      "Phishy",
      "Suspicious",
      "Suspicious",
      "Legitimate",
      "Phishy",
      "Legitimate",
      "Legitimate",
      "Legitimate",
      "Legitimate",
      "Phishy",
      "Suspicious",
      "Suspicious",
      "Phishy",
      "Phishy",
      "Suspicious",
      "Legitimate",
      "Legitimate",
      "Suspicious",
      "Suspicious",
      "Suspicious",
      "Legitimate",
      "Legitimate",
      "Phishy",
      "Legitimate",
      "Legitimate",
      "Suspicious",
      "Phishy",
      "Legitimate",
      "Phishy",
      "Legitimate",
      "Leg

### Authenticate and predict
Use the running TabPFN deployment from Exercise 0. Credentials and tokens are never printed.


In [5]:
auth_url = os.environ["AICORE_AUTH_URL"].rstrip("/")
if not auth_url.endswith("/oauth/token"):
    auth_url += "/oauth/token"
auth = requests.post(auth_url, data={
    "grant_type": "client_credentials",
    "client_id": os.environ["AICORE_CLIENT_ID"],
    "client_secret": os.environ["AICORE_CLIENT_SECRET"],
}, timeout=60)
auth.raise_for_status()
headers = {
    "Authorization": "Bearer " + auth.json()["access_token"],
    "AI-Resource-Group": os.getenv("AICORE_RESOURCE_GROUP", "default"),
}
url = (os.environ["BASE_DEPLOYMENT_URL"].rstrip("/") + "/"
       + os.environ["TABPFN_DEPLOYMENT_ID"] + "/predict")


In [6]:
for name, body in [("Miami prices (USD)", miami_body), ("Website classes", phishing_body)]:
    response = requests.post(url, headers=headers, json=body, timeout=300)
    response.raise_for_status()
    result = response.json()
    assert not result.get("error"), "SAP returned an API error"
    predictions = result["prediction"]
    assert len(predictions) == 5, "Expected one prediction per query row"
    print(name, predictions)


Miami prices (USD) [275650.40625, 1472958.0, 185511.15625, 204684.71875, 189550.5625]


Website classes ['Legitimate', 'Phishy', 'Legitimate', 'Legitimate', 'Legitimate']
